## tl;dr
P1/P2/P3의 로컬 OOF를 독립 재계산합니다. 모델을 학습하거나 공식 입력을 읽지 않습니다. PENDING은 완료가 아닙니다.

## Context & Methods
F1은 TP/FP/FN 합계, RMSE는 전체 SSE/행수로 계산합니다. 원본 반복 노출 검증이므로 fresh 또는 공식 점수로 해석하지 않습니다.

### Key Assumptions
러너가 기록한 key/truth 대응과 split/provenance는 별도 코드 QA가 필요합니다. 이 노트북은 수치 검산을 담당합니다.

In [1]:
from pathlib import Path
import sys
repo = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'AGENTS.md').is_file()), None)
if repo is None:
    raise RuntimeError('Run inside the research repository')
sys.path.insert(0, str(repo / 'scripts'))
from qa_parallel_score_repair_20260905_v1 import audit_run


## Data
입력은 ignored `artifacts/p?_score_repair_20260905_v1/qa_oof.npz` 세 파일뿐입니다. 관측/정답 행은 출력하지 않습니다.

In [2]:
audits = {problem: audit_run(problem, repo) for problem in ('P1', 'P2', 'P3')}
[{ 'problem': p, 'status': a['status'], 'rows': a.get('rows'), 'archive_sha256': a.get('archive_sha256') } for p, a in audits.items()]

[{'problem': 'P1',
  'status': 'NUMERICAL_QA_PASS',
  'rows': 421032,
  'archive_sha256': '94bd1e79ff28fe6bdb7bac35232e5ccde74cf6dff65cc14c597ee476f43274f8'},
 {'problem': 'P2',
  'status': 'NUMERICAL_QA_PASS',
  'rows': 69850,
  'archive_sha256': '41ebd6a7b0e994e977796bda1792fddec5ac4bfb44119db64e9a04ee2d264b2c'},
 {'problem': 'P3',
  'status': 'NUMERICAL_QA_PASS',
  'rows': 1086,
  'archive_sha256': '2eea67420fcb92e2fb0b6f570c4f09676929a5aee94bd49b52c01cd460b40b78'}]

## Results
후보−기준: F1은 양수, RMSE는 음수일 때 개선입니다. 동일 후보의 순수 수치 검산이며 제출 자격 전체를 자동 인증하지 않습니다.

In [3]:
[{ 'problem': p, 'status': a['status'], 'reference': a.get('reference'), 'candidate': a.get('candidate'), 'delta': a.get('candidate_minus_reference'), 'changed_rows': a.get('changed_rows') } for p, a in audits.items()]

[{'problem': 'P1',
  'status': 'NUMERICAL_QA_PASS',
  'reference': {'f1': 0.8511742399041979, 'tp': 12794, 'fp': 1213, 'fn': 3261},
  'candidate': {'f1': 0.8369278813311213, 'tp': 12864, 'fp': 1822, 'fn': 3191},
  'delta': -0.014246358573076656,
  'changed_rows': 1031},
 {'problem': 'P2',
  'status': 'NUMERICAL_QA_PASS',
  'reference': {'rmse': 0.92034593873515,
   'sse': 59165.509789197975,
   'rows': 69850,
   'bias': -0.2486724759412857},
  'candidate': {'rmse': 0.8592499137102484,
   'sse': 51570.982432643184,
   'rows': 69850,
   'bias': -0.13941828402255294},
  'delta': -0.061096025024901635,
  'changed_rows': 69850},
 {'problem': 'P3',
  'status': 'NUMERICAL_QA_PASS',
  'reference': {'rmse': 0.7791048399763751,
   'sse': 659.2067259186298,
   'rows': 1086,
   'bias': -0.10339048747292207},
  'candidate': {'rmse': 0.7783397516664796,
   'sse': 657.9126671603215,
   'rows': 1086,
   'bias': -0.10975141079233083},
  'delta': -0.000765088309895523,
  'changed_rows': 181}]

## Takeaways
미완료 결과를 임의로 채우지 않습니다. 공식 점수와 재현성/출처 QA는 별도 영수증을 따릅니다.

In [4]:
pending = [p for p, a in audits.items() if a['status'] != 'NUMERICAL_QA_PASS']
print('Numerical verification complete' if not pending else 'Incomplete: ' + ', '.join(pending))

Numerical verification complete
